In [1]:
import pandas as pd
import numpy as np

# 1. Carregar a base original (Morumbi)

In [2]:
# Carrega o arquivo de entrada (arquivo organizado em `data/filling_Ceps/`)
df = pd.read_csv('../../data/filling_Ceps/Elvira Brandão Morumbi - Euvira Brandão Dados ADS_coords_corrigidas_com_enderecos.csv')

# 2. Tratar a renda para número

In [3]:
import re

def parse_renda_seguro(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    s = re.sub(r"[^0-9,\.]", "", s)
    if "," in s:
        s = s.replace(".", "")
    elif s.count(".") > 1:
        s = s.replace(".", "")
    s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan


df["renda_media_num"] = df["renda_media"].apply(parse_renda_seguro)


# 3. Criar Total_0_9 (0–4 + 5–9 anos)

In [4]:
df["Total_0_9"] = (
    df["v01031_0_4anos"].fillna(0) +
    df["v01032_5_9anos"].fillna(0)
)

In [5]:
import math

def haversine_km(lat, lon, lat0, lon0):
    lat = np.radians(pd.to_numeric(lat, errors="coerce"))
    lon = np.radians(pd.to_numeric(lon, errors="coerce"))
    lat0 = math.radians(lat0)
    lon0 = math.radians(lon0)
    dlat = lat - lat0
    dlon = lon - lon0
    a = np.sin(dlat / 2) ** 2 + np.cos(lat0) * np.cos(lat) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0 * c

# Coordenadas da unidade
escola_lat = -23.6164
escola_lon = -46.73831

# Distancia por ponto
if "latitude_centro" in df.columns and "longitude_centro" in df.columns:
    df["distancia_km"] = haversine_km(df["latitude_centro"], df["longitude_centro"], escola_lat, escola_lon)
else:
    df["distancia_km"] = np.nan

# Parametros da filtragem por robustez e tamanho das listas
min_pontos = 5
top_n = 12

# 4. Corrigir renda para 2025 (inflação)

In [6]:
inflation_factor = 1.155
df["renda_atualizada_2025"] = df["renda_media_num"] * inflation_factor

# 5. Score linha a linha (informativo, não consolidado)

In [7]:
df["score_trafego_2025"] = df["renda_atualizada_2025"] * df["Total_0_9"]

# 6. Consolidar por CEP (SEM SOMAR — usar média!)

In [8]:
df_cep = (
    df.groupby("CEP", as_index=False)
      .agg({
          "Bairro": lambda x: x.mode().iat[0] if not x.mode().empty else x.iloc[0],
          "renda_atualizada_2025": "median",
          "Total_0_9": "median",
          "populacao_total": "median",
          "distancia_km": "median",
      })
)

pontos = df.groupby("CEP").size().reset_index(name="pontos")
df_cep = df_cep.merge(pontos, on="CEP", how="left")


In [9]:
# Renomear colunas para refletir que sao medianas

df_cep = df_cep.rename(columns={
    "renda_atualizada_2025": "renda_mediana_2025",
    "Total_0_9": "mediana_criancas_0_9",
    "populacao_total": "populacao_mediana",
    "distancia_km": "distancia_mediana_km",
})


# 7. Score final no nível do CEP (correto)

In [10]:
df_cep["score_trafego_2025"] = (
    df_cep["renda_mediana_2025"] * df_cep["mediana_criancas_0_9"]
)

# 8. Ranking final

In [11]:
top_ceps = df_cep.sort_values("score_trafego_2025", ascending=False)

print(top_ceps.head(15))

           CEP                            Bairro  renda_mediana_2025  \
512  05635-050                Jardim Monte Kemel         24449.50200   
103  04719-905  Chácara Santo Antônio (Zona Sul)         32647.68045   
667  05709-040                       Vila Suzana         36272.94825   
134  04729-060                  Jardim Caravelas         19448.42130   
42   04583-909                     Vila Cordeiro         28934.10135   
118  04726-160                     Vila Cruzeiro         24868.25880   
617  05679-050           Jardim Panorama D'Oeste         35331.79650   
501  05634-001                Jardim Monte Kemel         22594.04070   
725  05726-140                      Vila Andrade         19572.22575   
16   04567-002                    Cidade Monções         27318.29100   
136  04730-000                   Várzea de Baixo         18074.72205   
664  05707-400                 Parque do Morumbi         21891.88155   
67   04709-901                       Santo Amaro         25861.4

In [12]:
# Arredondar para 2 casas decimais (padrao monetario)
top_ceps["renda_mediana_2025"] = top_ceps["renda_mediana_2025"].round(2)
top_ceps["score_trafego_2025"] = top_ceps["score_trafego_2025"].round(2)
if "distancia_mediana_km" in top_ceps.columns:
    top_ceps["distancia_mediana_km"] = top_ceps["distancia_mediana_km"].round(2)


In [13]:
# Salvar resultado agregado na pasta do notebook (comportamento original)
top_ceps.to_csv("morumbi_top_ceps_2025.csv", index=False)

In [14]:
# Filtrar por robustez (sem filtro por bairro)
top_ceps_filtrado = top_ceps[top_ceps["pontos"] >= min_pontos].copy()

top_ceps_filtrado


,CEP,Bairro,renda_mediana_2025,mediana_criancas_0_9,populacao_mediana,distancia_mediana_km,pontos,score_trafego_2025
724,05726-130,Vila Andrade,11951.80,92.0,600.0,1.00,5,1099565.73
660,05706-290,Paraíso do Morumbi,25705.64,27.0,296.0,1.96,6,694052.27
534,05641-030,Vila Suzana,21049.40,27.0,342.0,0.25,7,568333.84
672,05711-001,Jardim Caboré,12730.75,44.0,367.0,0.72,6,560153.03
751,05734-080,Vila Andrade,9578.44,56.0,390.0,1.52,5,536392.53
531,05641-010,Vila Suzana,21478.15,24.0,292.0,0.43,5,515475.58
706,05717-270,Vila Andrade,15458.43,31.0,357.0,2.20,5,479211.26
662,05707-000,Parque do Morumbi,21933.32,20.0,188.0,2.58,5,438666.46
731,05727-240,Vila Andrade,12706.02,34.0,237.0,1.05,5,432004.56
687,05713-520,Jardim Ampliação,15629.63,27.5,334.5,1.49,6,429814.76


In [15]:
top_ceps_filtrado.to_csv("morumbi_top_ceps_filtrados_2025.csv", index=False)